# Notebook 01 — Project Overview, Data Sources & Responsible Use
## Afghanistan Multi-Index Drought Stress Dashboard
---

## Research Objective

This project develops a Composite Drought Index (CDI) for Afghanistan at the district level using satellite-derived environmental indicators. The goal is to provide:
- A transparent, reproducible framework for monitoring environmental drought stress
- District-level analysis across 399 districts from 2000–2025
- Tools for geographic prioritization and seasonal pattern analysis

CDI Interpretation: The index is scaled 0–100, where:
- 0 = Extreme drought (vegetation stressed, high temperatures, low precipitation)
- 50 = Normal conditions
- 100 = Extremely wet/favorable conditions

---

## Analysis Pipeline Summary

| Notebook | Description |
|----------|-------------|
| 01 | Project overview, data sources, ethics, and limitations |
| 02 | Data extraction from Google Earth Engine, preprocessing, and spatial aggregation |
| 03 | Drought indicator construction (VCI, TCI, SPI-3) with quality masking and interpolation |
| 04 | Composite Drought Index (CDI) creation and weight sensitivity analysis |
| 05 | Mapping, validation, hotspot identification, and spatiotemporal analysis |

## Data Sources

Data is sourced from Google Earth Engine and aggregated to Afghanistan's 399 districts.

| Variable | Source | Temporal Resolution | Processing |
|----------|--------|---------------------|------------|
| NDVI (Vegetation) | MODIS MOD13A3 | Monthly | Mean, QA-masked (SummaryQA ≤ 1) |
| LST (Land Surface Temperature) | MODIS MOD11A2 | 8-day → Monthly | Mean, QA-masked, converted to °C |
| Precipitation | CHIRPS Daily | Daily → Monthly | Sum (mm/month) |
| District boundaries | FAO GAUL 2015 Level 2 | Static | 399 districts |

Study Period: March 2000 – November 2025 (309 months)

Baseline Period for Climatology: 2001–2020 (used for percentile calculations in VCI/TCI and gamma fitting for SPI)

## Drought Indices Constructed

### 1. VCI (Vegetation Condition Index)

$$\text{VCI} = 100 \times \frac{\text{NDVI}_{\text{current}} - \text{NDVI}_\text{min}}{\text{NDVI}_\text{max} - \text{NDVI}_\text{min}}$$

- Measures vegetation health relative to historical conditions for the same month
- Uses 5th and 95th percentiles from the 2001–2020 baseline for NDVI min/max
- Interpretation: VCI = 0 → worst vegetation observed; VCI = 100 → best vegetation observed
- Values outside [0, 100] are clipped for robustness

### 2. TCI (Temperature Condition Index)

$$\text{TCI} = 100 \times \frac{\text{LST}_\text{max} - \text{LST}_{\text{current}}}{\text{LST}_\text{max} - \text{LST}_\text{min}}$$

- Measures heat stress relative to historical conditions for the same month
- Uses 5th and 95th percentiles from the 2001–2020 baseline for LST min/max
- Interpretation: TCI = 0 → extreme heat stress; TCI = 100 → coolest conditions observed
- Values outside [0, 100] are clipped

### 3. SPI-3 (Standardized Precipitation Index, 3-month)

- Measures precipitation anomalies using a gamma distribution fit
- Interpretation: SPI = −3 → extreme drought; SPI = 0 → normal; SPI = +3 → extremely wet
- Normalized scale: 0 = extreme drought, 50 = normal, 100 = extremely wet

Process:
1. Compute 3-month rolling precipitation sums per district
2. Fit gamma distribution to the baseline period (2001–2020) for each district-month
3. Transform to standard normal using CDF: $\text{SPI} = \Phi^{-1}(\text{Gamma CDF}(P))$
4. Normalize to 0–100 scale: $\text{SPI}_{\text{norm}} = 50 + \text{SPI} \times 16.67$

## Composite Drought Index (CDI)

The CDI combines all three indicators into a single drought stress metric. Equal weights are used as a common baseline; an alternative weighting scheme tests robustness. Determining optimal weights is outside this project's scope. Future work could incorporate ground-truth data or Principal Component Analysis.

Primary CDI (Equal Weights): $$\text{CDI} = \frac{1}{3} \times \text{VCI} + \frac{1}{3} \times \text{TCI} + \frac{1}{3} \times \text{SPI}_{\text{norm}}$$

Alternative CDI (VCI-Heavy):
$$\text{CDI}_{\text{alt}} = 0.5 \times \text{VCI} + 0.3 \times \text{TCI} + 0.2 \times \text{SPI}_{\text{norm}}$$

Partial Composites: When SPI failed to compute (0.44% of data due to gamma fit failures in arid regions), CDI used only VCI and TCI with adjusted weights:
- CDI: 0.5 × VCI + 0.5 × TCI
- CDI_alt: 0.625 × VCI + 0.375 × TCI

### Why Clip Values?
VCI and TCI use 5th/95th percentiles (not absolute min/max), so ~10% of values naturally fall outside [0, 100]. Clipping ensures all components share a bounded scale, improves interpretability, and prevents extreme outliers from dominating the composite. Raw unclipped values are preserved in the `VCI` and `TCI` columns with quality flags (`VCI_out_of_range`, `TCI_out_of_range`) for transparency.

## Data Quality & Preprocessing Decisions

### Missing Data Handling
| Issue | Count | Approach |
|-------|-------|----------|
| Missing NDVI (cloud cover) | 535 points (0.43%) | Time-based interpolation |
| Missing LST | 37 points (0.03%) -> negligible| Time-based interpolation |
| SPI gamma fit failures | 475 points (0.44%) | Partial CDI using VCI + TCI only |

NDVI Missing Pattern: 99.8% of missing NDVI occurred in January–February in mountainous districts due to winter cloud cover. All gaps were ≤2 months, suitable for interpolation.

SPI Gamma Fit Failures: 100% of SPI failures occur in September. SPI-3 for September uses July–September precipitation; 19 arid districts consistently have near-zero summer rainfall, causing gamma fitting to fail.


### Quality Flags
Each observation includes quality flags for transparency:
- `NDVI_was_filled`: NDVI was interpolated
- `LST_was_filled`: LST was interpolated  
- `VCI_out_of_range`: Raw VCI was <0 or >100 (clipped for analysis)
- `TCI_out_of_range`: Raw TCI was <0 or >100 (clipped for analysis)
- `CDI_partial`: CDI computed without SPI (gamma fit failed)
- `any_quality_flag`: At least one quality concern

Overall Quality: 25.2% of observations had at least one flag. VCI and TCI out-of-range flags account for 97.4% of flagged observations, which is expected given that 5th/95th percentiles produce at least 10% out-of-range values per index.

---
## Key Findings

Weight Sensitivity: CDI weighting schemes showed strong agreement (r = 0.975), confirming robustness.

Validation: CDI captured documented drought periods: 2000–2002, 2008, 2018, and 2021–2022.

Spatial Patterns:
- Most drought-prone districts concentrated in southwestern Afghanistan
- Most volatile districts: Ghazni Province (5 of top 10)
- Clear seasonal patterns visible in province-by-month heatmaps

## What This Analysis Can (and Cannot) Inform

Appropriate uses:
- Geographic prioritization of drought-prone districts
- Strategic planning for climate adaptation and drought preparedness
- Early screening to flag areas needing deeper field assessment
- Cross-sector dialogue alongside livelihood, protection, or displacement data
- Historical pattern analysis over the 25-year record

Not appropriate for:
- Predicting migration, displacement, or humanitarian needs
- Inferring household-level vulnerability or food security
- Attributing causality between drought and social outcomes
- Replacing field data or community consultation

## Limitations & Responsible Use

Responsible use:
- District-scale aggregation only (no household inference)
- All methods and assumptions documented transparently
- Use alongside qualitative research, not as replacement
- Avoid environmental determinism as drought stress is one factor among many

Key limitations:
- Satellite proxies are not direct measures of agricultural impact
- Equal weighting is arbitrary (sensitivity analysis shows robustness)
- ~0.44% of SPI values failed due to near-zero precipitation in arid districts
- 20-year baseline is shorter than ideal
- Validation was qualitative (visual comparison with SPEI, alignment with known drought events); quantitative validation against ground-truth crop or livelihood data would strengthen confidence

The CDI is a screening and triangulation tool, not a standalone diagnostic.

## CDI Interpretation Guide

| CDI Range | Category |
|-----------|----------|
| 0–10 | Extreme drought |
| 10–20 | Severe drought |
| 20–30 | Moderate drought |
| 30–40 | Mild drought |
| 40–50 | Near normal (dry) |
| 50 | Normal |
| 50–100 | Wet conditions |

Thresholds follow established VCI/TCI conventions and align with standard SPI categories when normalized.

## Output Files

| Location | File | Description |
|----------|------|-------------|
| `data/processed/` | `afg_drought_indicators_2000_2025.csv` | Full dataset with VCI, TCI, SPI, CDI, and quality flags |
| `data/processed/` | `afg_district_drought_summary.csv` | District-level summary statistics |
| `data/interim/` | `*_parameters.csv` | VCI, TCI, and SPI calibration parameters |
| `data/raw/` | `afg_climate_2000_2025.csv` | Raw NDVI, LST, and precipitation from GEE |

---

## Google Earth Engine Connectivity Check

Optional, only needed if re-running data extraction in Notebook 02. 

In [1]:
# Earth Engine connectivity check
try:
    import ee
    ee.Initialize()

    afg = (ee.FeatureCollection("FAO/GAUL/2015/level0")
           .filter(ee.Filter.eq("ADM0_NAME", "Afghanistan")))

    print(f"Earth Engine Initialized - Afghanistan features: {afg.size().getInfo()}")
except Exception as e:
    print(f"Earth Engine not available: {e}")

Earth Engine Initialized - Afghanistan features: 1
